# 🎯 Fazlerasheed Vision — GPU-Accelerated Activity Monitor
### Google Colab Edition

Full **YOLOv8n + ByteTrack + InsightFace + GPT-4o** person-activity pipeline running on a **Colab GPU** with **live webcam** from your browser.

| Stage | Component | Hardware |
|-------|-----------|----------|
| Detection + Tracking | YOLOv8n + ByteTrack | **GPU** |
| Face Recognition | InsightFace Buffalo_L | **GPU** |
| Activity Narration | Azure / Gemini / OpenAI | Cloud API |
| Event Log | SQLite | Google Drive |

### ✅ Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. Fill in API keys in **Cell 3 – Configuration**
3. Run all cells top-to-bottom (Runtime → Run all)


## 1 · GPU Check

In [ ]:
import subprocess, torch

print("=" * 55)
print("  GPU / Runtime Info")
print("=" * 55)
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  ✅  GPU  : {name}")
    print(f"  💾  VRAM : {mem:.1f} GB")
else:
    print("  ⚠️   No GPU detected — switch runtime to GPU!")
    print("       Runtime → Change runtime type → T4 GPU")
print("=" * 55)
print(f"  Python : {subprocess.check_output('python --version', shell=True).decode().strip()}")
print(f"  PyTorch: {torch.__version__}")
print("=" * 55)


## 2 · Install Dependencies

In [ ]:
# Install all dependencies (takes ~2 min on first run)
%pip install -q ultralytics insightface onnxruntime-gpu openai \
             google-generativeai python-dotenv ipywidgets

print("\n✅ All packages installed.")


## 3 · Configuration
> Fill in your API keys below before proceeding.

In [ ]:
# ─────────────────────────────────────────────────────────
#  ⚙️  CONFIGURATION  — fill in your keys here
# ─────────────────────────────────────────────────────────

# ── LLM Provider: "azure" | "gemini" | "openai" | "disabled"
LLM_PROVIDER = "azure"

# ── Azure OpenAI ─────────────────────────────────────────
AZURE_OPENAI_API_KEY    = ""   # your Azure API key
AZURE_OPENAI_ENDPOINT   = "https://YOUR-RESOURCE.openai.azure.com/"
AZURE_OPENAI_DEPLOYMENT = "gpt-4o"
AZURE_OPENAI_API_VERSION = "2024-08-01-preview"

# ── Google Gemini ─────────────────────────────────────────
GEMINI_API_KEY = ""   # AIzaSy...

# ── OpenAI Direct ─────────────────────────────────────────
OPENAI_API_KEY = ""   # sk-...

# ── Narration tuning ──────────────────────────────────────
NARRATE_INTERVAL_S    = 5    # seconds between LLM calls per person
BOX_PAD_PX            = 20   # padding around person crop
BLUR_FACES_BEFORE_SEND = False

# ── Paths ─────────────────────────────────────────────────
import os
DRIVE_BASE       = "/content/drive/MyDrive/fazlerasheed_vision"
KNOWN_FACES_DIR  = os.path.join(DRIVE_BASE, "known_faces")
DB_PATH          = os.path.join(DRIVE_BASE, "events.db")
ENCODINGS_FILE   = os.path.join(KNOWN_FACES_DIR, "encodings.pkl")

print("✅ Configuration loaded.")
print(f"   LLM Provider : {LLM_PROVIDER}")
print(f"   Data path    : {DRIVE_BASE}")


## 4 · Mount Google Drive
Saves `events.db` and `known_faces/encodings.pkl` across sessions.

In [ ]:
# Mount Google Drive (persists known_faces/ and events.db across sessions)
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(KNOWN_FACES_DIR, exist_ok=True)
print(f"\n✅ Drive mounted. Data will be saved to:\n   {DRIVE_BASE}")


## 5 · Load Core Modules
All modules are inlined below — no separate `.py` files needed.

In [ ]:
# ── Logger module ─────────────────────────────────────────
import sqlite3
from datetime import datetime, date

_CREATE_TABLE = '''
CREATE TABLE IF NOT EXISTS events (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp  TEXT    NOT NULL,
    event_type TEXT    NOT NULL,
    person_id  TEXT,
    extra      TEXT
);
'''

class EventLogger:
    def __init__(self, db_path=DB_PATH):
        self._db_path = db_path
        self._conn = sqlite3.connect(db_path, check_same_thread=False)
        self._conn.execute(_CREATE_TABLE)
        self._conn.commit()
        print(f"[EventLogger] Database: {db_path}")

    def _write(self, event_type, person_id=None, extra=None):
        ts = datetime.now().isoformat(timespec="seconds")
        self._conn.execute(
            "INSERT INTO events (timestamp, event_type, person_id, extra) VALUES (?,?,?,?)",
            (ts, event_type, person_id, extra),
        )
        self._conn.commit()

    def person_detected(self, count=1):   self._write("person_detected",  extra=f"count={count}")
    def face_recognized(self, name):      self._write("face_recognized",  person_id=name)
    def unknown_face(self):               self._write("unknown_face")
    def activity_detected(self, label, person_id=None):
        self._write("activity_detected", person_id=person_id, extra=label)
    def high_priority_alert(self, detail): self._write("high_priority", extra=detail)
    def llm_narration(self, description): self._write("llm_narration", extra=description)

    def today_summary(self):
        today = date.today().isoformat()
        cur = self._conn.execute(
            "SELECT event_type, COUNT(*) FROM events WHERE timestamp LIKE ? GROUP BY event_type",
            (f"{today}%",))
        return {et: cnt for et, cnt in cur.fetchall()}

    def today_narrations(self):
        today = date.today().isoformat()
        cur = self._conn.execute(
            "SELECT timestamp, person_id, extra FROM events "
            "WHERE event_type='activity_detected' AND timestamp LIKE ? ORDER BY id ASC",
            (f"{today}%",))
        return [{"timestamp": r[0], "person_id": r[1], "extra": r[2]} for r in cur.fetchall()]

    def today_high_priority(self):
        today = date.today().isoformat()
        cur = self._conn.execute(
            "SELECT timestamp, extra FROM events "
            "WHERE event_type='high_priority' AND timestamp LIKE ? ORDER BY id ASC",
            (f"{today}%",))
        return [{"timestamp": r[0], "detail": r[1]} for r in cur.fetchall()]

    def recent_events(self, limit=20):
        cur = self._conn.execute(
            "SELECT timestamp, event_type, person_id, extra FROM events ORDER BY id DESC LIMIT ?",
            (limit,))
        cols = ["timestamp", "event_type", "person_id", "extra"]
        return [dict(zip(cols, row)) for row in cur.fetchall()]

    def close(self): self._conn.close()

print("✅ EventLogger defined.")


In [ ]:
# ── Tracker module (YOLOv8n + ByteTrack, GPU) ─────────────
import cv2
import numpy as np
import torch
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_PERSON_CLASS_ID      = 0
_CONFIDENCE_THRESHOLD = 0.40
_MODEL_PATH           = "yolov8n.pt"

_PALETTE = [
    (0, 180, 255), (0, 255, 120), (255, 80, 0),  (180, 0, 255),
    (255, 220, 0), (0, 220, 220), (255, 0, 140), (100, 255, 0),
    (0, 100, 255), (255, 140, 0), (0, 255, 200), (200, 0, 200),
    (255, 60, 60), (60, 255, 60), (60, 60, 255), (200, 200, 0),
]

def _id_colour(track_id):
    return _PALETTE[track_id % len(_PALETTE)]

class TrackResult:
    __slots__ = ("track_id", "box", "conf")
    def __init__(self, track_id, box, conf):
        self.track_id = track_id
        self.box      = box
        self.conf     = conf

class PersonTracker:
    def __init__(self, model_path=_MODEL_PATH):
        print(f"[PersonTracker] Loading YOLOv8n + ByteTrack on {DEVICE}...")
        self._model    = YOLO(model_path)
        self._prev_ids = set()
        print("[PersonTracker] Ready.")

    def update(self, frame):
        results = self._model.track(
            frame,
            classes  = [_PERSON_CLASS_ID],
            conf     = _CONFIDENCE_THRESHOLD,
            tracker  = "bytetrack.yaml",
            persist  = True,
            verbose  = False,
            device   = DEVICE,
        )
        tracks      = []
        current_ids = set()
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                if box.id is None:
                    continue
                track_id        = int(box.id[0])
                conf            = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                current_ids.add(track_id)
                tracks.append(TrackResult(track_id, (x1, y1, x2, y2), conf))
                colour = _id_colour(track_id)
                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
                cv2.putText(frame, f"ID {track_id}  {conf:.0%}",
                            (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2)
        entered = current_ids - self._prev_ids
        left    = self._prev_ids  - current_ids
        for tid in entered: print(f"[Tracker] Person ENTERED  ID={tid}")
        for tid in left:    print(f"[Tracker] Person LEFT     ID={tid}")
        self._prev_ids = current_ids
        return tracks

print(f"✅ PersonTracker defined  (will run on {DEVICE}).")


In [ ]:
# ── Recognizer module (InsightFace Buffalo_L, GPU) ─────────
import os, pickle
import numpy as np
import cv2

_SIMILARITY_THRESHOLD = 0.4
_ENROLL_PHOTOS        = 5

def _load_insightface():
    try:
        from insightface.app import FaceAnalysis
        import torch
        providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                     if torch.cuda.is_available() else ["CPUExecutionProvider"])
        app = FaceAnalysis(name="buffalo_l", providers=providers)
        app.prepare(ctx_id=0, det_size=(640, 640))
        print(f"[Recognizer] InsightFace loaded (providers={providers[:1]})")
        return app
    except ImportError as e:
        print(f"[Recognizer] InsightFace not available: {e}")
        return None

def _cosine_distance(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return 1.0 - float(np.dot(a, b))

class FaceRecognizer:
    def __init__(self):
        os.makedirs(KNOWN_FACES_DIR, exist_ok=True)
        self._app   = _load_insightface()
        self._known = self._load_known()

    def _load_known(self):
        if os.path.exists(ENCODINGS_FILE):
            with open(ENCODINGS_FILE, "rb") as f:
                known = pickle.load(f)
            print(f"[Recognizer] Loaded {len(known)} known person(s): {list(known.keys())}")
            return known
        print("[Recognizer] No known faces yet — enroll someone first.")
        return {}

    def _save_known(self):
        with open(ENCODINGS_FILE, "wb") as f:
            pickle.dump(self._known, f)

    def identify(self, frame):
        results = []
        if self._app is None:
            return results, frame
        faces = self._app.get(frame)
        for face in faces:
            box  = face.bbox.astype(int)
            emb  = face.embedding
            name = "Unknown"
            best = _SIMILARITY_THRESHOLD
            for kname, kemb in self._known.items():
                d = _cosine_distance(emb, kemb)
                if d < best:
                    best, name = d, kname
            color = (0, 220, 80) if name != "Unknown" else (0, 60, 220)
            cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), color, 2)
            label = name if name == "Unknown" else f"{name} ({1-best:.0%})"
            cv2.putText(frame, label, (box[0], box[1] - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            results.append({"name": name,
                             "box": tuple(box[:4].tolist()),
                             "distance": float(best)})
        return results, frame

    def enroll_from_frames(self, name, frames):
        """Enroll `name` from a list of numpy BGR frames."""
        if self._app is None:
            print("[Recognizer] InsightFace not available.")
            return False
        embeddings = []
        for frame in frames:
            faces = self._app.get(frame)
            if faces:
                embeddings.append(faces[0].embedding.copy())
        if not embeddings:
            print("[Recognizer] No faces detected in provided frames.")
            return False
        self._known[name] = np.mean(embeddings, axis=0)
        self._save_known()
        print(f"[Recognizer] '{name}' enrolled from {len(embeddings)} frame(s).")
        return True

print("✅ FaceRecognizer defined.")


In [ ]:
# ── Narrator module (LLM activity narration) ──────────────
import base64, time, threading, re
import cv2
import numpy as np

_HIGH_PRIORITY_KEYWORDS = [
    "drawer", "drawers", "fridge", "refrigerator",
    "cabinet", "cabinets", "cupboard",
]

_ACTIVITY_PROMPT = (
    "You are a security monitoring assistant analysing a cropped image of one person. "
    "In a single short phrase (5 words or fewer), describe the most likely action this person is performing. "
    "Choose the best match from this list if applicable: "
    "standing, sitting, bending, drinking, eating, talking, opening a fridge, opening a drawer. "
    "If none fit, give your own brief description. "
    "Do NOT identify the person by name. Do NOT add extra commentary."
)

_SUMMARY_PROMPT_TEMPLATE = (
    "You are generating an end-of-shift activity report. "
    "Below are timestamped per-person activity descriptions. "
    "Write a concise, professional plain-English summary (3-5 sentences). "
    "Note any notable events, patterns, or high-priority alerts.\n\n"
    "Activity log:\n{narrations}\n\nEvent counts:\n{event_counts}"
)

def _frame_to_b64(frame, quality=75):
    _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return base64.b64encode(buf.tobytes()).decode()

def _has_high_priority(text):
    lower = text.lower()
    for kw in _HIGH_PRIORITY_KEYWORDS:
        if re.search(r'\b' + re.escape(kw) + r'\b', lower):
            return kw
    return None

def _blur_faces(frame):
    try:
        cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = cascade.detectMultiScale(gray, 1.1, 4)
        out   = frame.copy()
        for (x, y, fw, fh) in faces:
            out[y:y+fh, x:x+fw] = cv2.GaussianBlur(out[y:y+fh, x:x+fw], (51, 51), 0)
        return out
    except Exception:
        return frame

def _call_azure(b64):
    from openai import AzureOpenAI
    client = AzureOpenAI(
        api_key        = AZURE_OPENAI_API_KEY,
        azure_endpoint = AZURE_OPENAI_ENDPOINT,
        api_version    = AZURE_OPENAI_API_VERSION,
    )
    r = client.chat.completions.create(
        model    = AZURE_OPENAI_DEPLOYMENT,
        messages = [{"role": "user", "content": [
            {"type": "text",      "text": _ACTIVITY_PROMPT},
            {"type": "image_url", "image_url": {
                "url": f"data:image/jpeg;base64,{b64}", "detail": "low"}},
        ]}],
        max_tokens = 60,
    )
    return r.choices[0].message.content.strip()

_GEMINI_MODELS = ["gemini-2.0-flash", "gemini-1.5-flash-latest", "gemini-1.5-flash"]

def _call_gemini(b64):
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    for model_name in _GEMINI_MODELS:
        try:
            model = genai.GenerativeModel(model_name)
            r = model.generate_content([_ACTIVITY_PROMPT,
                                        {"mime_type": "image/jpeg", "data": b64}])
            return r.text.strip()
        except Exception as e:
            if "404" in str(e) or "not found" in str(e).lower():
                continue
            return f"[Narrator] Gemini error: {e}"
    return "[Narrator] No Gemini model available."

def _call_openai(b64):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    r = client.chat.completions.create(
        model    = "gpt-4o-mini",
        messages = [{"role": "user", "content": [
            {"type": "text",      "text": _ACTIVITY_PROMPT},
            {"type": "image_url", "image_url": {
                "url": f"data:image/jpeg;base64,{b64}", "detail": "low"}},
        ]}],
        max_tokens = 60,
    )
    return r.choices[0].message.content.strip()

def _describe_crop(crop):
    if LLM_PROVIDER == "disabled":
        return ""
    b64 = _frame_to_b64(_blur_faces(crop) if BLUR_FACES_BEFORE_SEND else crop)
    try:
        if LLM_PROVIDER == "azure":  return _call_azure(b64)
        if LLM_PROVIDER == "gemini": return _call_gemini(b64)
        if LLM_PROVIDER == "openai": return _call_openai(b64)
    except Exception as e:
        return f"[Narrator] API error: {e}"
    return f"[Narrator] Unknown provider: {LLM_PROVIDER}"

def _crop_box(frame, box, pad=BOX_PAD_PX):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = box
    return frame[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)]

class Narrator:
    def __init__(self, logger):
        self._log          = logger
        self._last_narrate = {}
        self._lock         = threading.Lock()
        self._active       = LLM_PROVIDER != "disabled"
        if self._active:
            print(f"[Narrator] Ready. Provider={LLM_PROVIDER}  "
                  f"interval={NARRATE_INTERVAL_S}s  blur={BLUR_FACES_BEFORE_SEND}")
        else:
            print("[Narrator] Disabled (LLM_PROVIDER=disabled).")

    def maybe_narrate(self, frame, tracks, force=False):
        if not self._active or not tracks:
            return
        now = time.time()
        for t in tracks:
            last = self._last_narrate.get(t.track_id, 0.0)
            if force or (now - last) >= NARRATE_INTERVAL_S:
                self._last_narrate[t.track_id] = now
                crop = _crop_box(frame, t.box)
                if crop.size == 0:
                    continue
                threading.Thread(
                    target=self._narrate_async,
                    args=(crop.copy(), t.track_id),
                    daemon=True,
                ).start()

    def _narrate_async(self, crop, track_id):
        with self._lock:
            print(f"[Narrator] Describing person ID={track_id}…")
            desc = _describe_crop(crop)
            if not desc:
                return
            self._log.activity_detected(label=desc, person_id=str(track_id))
            print(f"[Narrator] ID={track_id}: {desc}")
            kw = _has_high_priority(desc)
            if kw:
                msg = f"HIGH PRIORITY — ID={track_id} may be '{kw}': {desc}"
                print(f"\n{'='*55}\n  ⚠  ALERT: {msg}\n{'='*55}\n")
                self._log.high_priority_alert(msg)

    def generate_summary(self):
        if not self._active:
            return "Stage 5 not active."
        narrations = self._log.today_narrations()
        if not narrations:
            return "No narrations recorded yet."
        nar_text  = "\n".join(
            f"  [{r['timestamp']}] Person {r['person_id']}: {r['extra']}"
            for r in narrations)
        cnt_text  = "\n".join(f"  {k}: {v}"
                               for k, v in self._log.today_summary().items())
        prompt = _SUMMARY_PROMPT_TEMPLATE.format(
            narrations=nar_text, event_counts=cnt_text)
        print("\n[Narrator] Generating shift summary…")
        try:
            if LLM_PROVIDER == "azure":
                from openai import AzureOpenAI
                client = AzureOpenAI(api_key=AZURE_OPENAI_API_KEY,
                                     azure_endpoint=AZURE_OPENAI_ENDPOINT,
                                     api_version=AZURE_OPENAI_API_VERSION)
                r = client.chat.completions.create(
                    model=AZURE_OPENAI_DEPLOYMENT,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=300)
                summary = r.choices[0].message.content.strip()
            elif LLM_PROVIDER == "gemini":
                import google.generativeai as genai
                genai.configure(api_key=GEMINI_API_KEY)
                for mn in _GEMINI_MODELS:
                    try:
                        summary = genai.GenerativeModel(mn).generate_content(prompt).text.strip()
                        break
                    except Exception:
                        continue
                else:
                    summary = "Gemini unavailable."
            elif LLM_PROVIDER == "openai":
                from openai import OpenAI
                r = OpenAI(api_key=OPENAI_API_KEY).chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=300)
                summary = r.choices[0].message.content.strip()
            else:
                summary = "Unknown provider."
        except Exception as e:
            summary = f"Summary failed: {e}"
        print(f"\n=== Shift Summary ===\n{summary}\n")
        self._log.llm_narration(f"[SHIFT SUMMARY] {summary}")
        return summary

print("✅ Narrator defined.")


## 6 · Webcam Utilities
JavaScript bridge for live webcam access inside Colab.

In [ ]:
# ── Webcam utilities (JavaScript ↔ Python bridge) ─────────
import base64
import numpy as np
import cv2
from google.colab.output import eval_js
from IPython.display import Javascript, display as ipy_display

_WEBCAM_INIT_JS = '''
(async () => {
    if (window._fr_webcam_ready) return 'already_ready';
    try {
        window._fr_stream = await navigator.mediaDevices.getUserMedia({
            video: {width: {ideal: 640}, height: {ideal: 480}, facingMode: 'user'}
        });
        window._fr_video = document.createElement('video');
        window._fr_video.srcObject = window._fr_stream;
        await window._fr_video.play();
        await new Promise(r => {
            if (window._fr_video.videoWidth > 0) { r(); return; }
            window._fr_video.onloadedmetadata = r;
        });
        window._fr_canvas = document.createElement('canvas');
        window._fr_canvas.width  = window._fr_video.videoWidth  || 640;
        window._fr_canvas.height = window._fr_video.videoHeight || 480;
        window._fr_webcam_ready = true;
        return 'ready:' + window._fr_canvas.width + 'x' + window._fr_canvas.height;
    } catch(e) {
        return 'error: ' + e.toString();
    }
})()
'''

_WEBCAM_CAPTURE_JS = '''
(function() {
    if (!window._fr_webcam_ready) return null;
    const ctx = window._fr_canvas.getContext('2d');
    ctx.drawImage(window._fr_video, 0, 0,
                  window._fr_canvas.width, window._fr_canvas.height);
    return window._fr_canvas.toDataURL('image/jpeg', 0.85);
})()
'''

_WEBCAM_STOP_JS = '''
(function() {
    if (window._fr_stream) {
        window._fr_stream.getTracks().forEach(t => t.stop());
    }
    window._fr_webcam_ready = false;
    return 'stopped';
})()
'''

def webcam_init():
    """Open the browser webcam. Must be called before capture."""
    result = eval_js(_WEBCAM_INIT_JS)
    print(f"[Webcam] {result}")
    return "ready" in str(result)

def webcam_capture():
    """Capture one frame. Returns BGR numpy array or None."""
    js_data = eval_js(_WEBCAM_CAPTURE_JS)
    if not js_data:
        return None
    img_bytes = base64.b64decode(js_data.split(',')[1])
    arr = np.frombuffer(img_bytes, dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

def webcam_stop():
    """Release the browser webcam."""
    result = eval_js(_WEBCAM_STOP_JS)
    print(f"[Webcam] {result}")

def frame_to_jpeg_bytes(frame, quality=85):
    """Encode BGR frame to JPEG bytes for ipywidgets display."""
    _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return buf.tobytes()

print("✅ Webcam utilities defined.")
print("   ▶  Run the next cell to open your browser webcam.")


## 7 · Face Enrollment
Run this cell once to save your face to Google Drive.
**Skip if** `encodings.pkl` already exists on Drive.

In [ ]:
# ── Face Enrollment ───────────────────────────────────────
# Captures ENROLL_N frames from your webcam and saves your face embedding.
# Skip this cell if you already have an encodings.pkl on Drive.

import time
import ipywidgets as widgets
from IPython.display import display as ipy_display

ENROLL_NAME = "qazi"   # ← change to your name
ENROLL_N    = 8        # number of frames to capture

print(f"Enrolling '{ENROLL_NAME}' — make sure your webcam is on (run webcam cell first).")

# Capture N frames with a short pause between them
frames = []
for i in range(ENROLL_N):
    f = webcam_capture()
    if f is not None:
        frames.append(f)
        print(f"  Captured frame {len(frames)}/{ENROLL_N}")
    time.sleep(0.4)

if frames:
    recognizer = FaceRecognizer()
    ok = recognizer.enroll_from_frames(ENROLL_NAME, frames)
    if ok:
        print(f"\n✅ '{ENROLL_NAME}' enrolled and saved to Drive.")
    else:
        print("\n❌ Enrollment failed — no faces detected. Check lighting/distance.")
else:
    print("❌ Could not capture any frames. Did you initialise the webcam?")


## 8 · Initialise Pipeline
Creates all components and the interactive control panel.

In [ ]:
# ── Initialise pipeline components ────────────────────────
import threading, time
import ipywidgets as widgets
from IPython.display import display as ipy_display

# Open webcam
print("Opening webcam...")
webcam_ok = webcam_init()
if not webcam_ok:
    print("\n⚠️  Webcam failed to open. Make sure you allow camera access in your browser.")

# Create components
log        = EventLogger()
tracker    = PersonTracker()
recognizer = FaceRecognizer()
narrator   = Narrator(log)

# ── UI widgets ─────────────────────────────────────────────
img_widget    = widgets.Image(format='jpeg', width=640, height=480)
status_label  = widgets.Label(value='Status: Ready — click ▶ Run Pipeline to start.')
fps_label     = widgets.Label(value='FPS: --')

stop_btn      = widgets.Button(description='⬛ Stop',       button_style='danger',
                               layout=widgets.Layout(width='110px'))
narrate_btn   = widgets.Button(description='🎤 Narrate',    button_style='info',
                               layout=widgets.Layout(width='110px'))
summary_btn   = widgets.Button(description='📊 Summary',    button_style='success',
                               layout=widgets.Layout(width='110px'))
alerts_btn    = widgets.Button(description='⚠ Alerts',     button_style='warning',
                               layout=widgets.Layout(width='110px'))

header = widgets.HTML(value='<h3 style="margin:4px 0">🎯 Fazlerasheed Vision — Live Feed</h3>')
btns   = widgets.HBox([stop_btn, narrate_btn, summary_btn, alerts_btn])
panel  = widgets.VBox([header, img_widget, widgets.HBox([status_label, fps_label]), btns],
                      layout=widgets.Layout(padding='10px'))
ipy_display(panel)

# ── Event flags ────────────────────────────────────────────
_stop          = threading.Event()
_force_narrate = threading.Event()
_gen_summary   = threading.Event()

def _on_stop(_):
    _stop.set()
    stop_btn.description = '⏹ Stopped'
    status_label.value   = 'Status: Stopped'

def _on_narrate(_):  _force_narrate.set()
def _on_summary(_):  _gen_summary.set()
def _on_alerts(_):
    alerts = log.today_high_priority()
    if alerts:
        for a in alerts:
            print(f"  ⚠  {a['timestamp']}  {a['detail']}")
    else:
        print("[Main] No high-priority alerts today.")

stop_btn.on_click(_on_stop)
narrate_btn.on_click(_on_narrate)
summary_btn.on_click(_on_summary)
alerts_btn.on_click(_on_alerts)

print("\n✅ Pipeline ready. Click ▶ on the next cell to start the live feed.")


## 9 · Run Live Pipeline
Click **▶ Run** below — the live feed will appear above in the widget.
Use the **⬛ Stop** button (rendered in the previous cell's output) to end the session.

In [ ]:
# ── Run the live pipeline ─────────────────────────────────
# ⚠  This cell BLOCKS until you click ⬛ Stop above.
# ─────────────────────────────────────────────────────────

RECOGNITION_EVERY_S = 1.0
INFERENCE_WIDTH     = 640

_stop.clear()
_force_narrate.clear()
_gen_summary.clear()
stop_btn.description = '⬛ Stop'

last_recog_time = 0.0
_seen_names     = {}
frame_count     = 0
t_start         = time.time()

print("[Main] Pipeline running…  Click ⬛ Stop to end.\n")

try:
    while not _stop.is_set():
        # ── 1. Capture frame from browser webcam ────────────
        frame = webcam_capture()
        if frame is None:
            time.sleep(0.05)
            continue

        now = time.time()
        frame_count += 1

        # ── 2. YOLO + ByteTrack (GPU) ────────────────────────
        h, w   = frame.shape[:2]
        scale  = INFERENCE_WIDTH / w if w > INFERENCE_WIDTH else 1.0
        if scale < 1.0:
            small  = cv2.resize(frame, (0, 0), fx=scale, fy=scale)
            tracks = tracker.update(small)
            for t in tracks:
                x1, y1, x2, y2 = t.box
                t.box = (int(x1/scale), int(y1/scale),
                         int(x2/scale), int(y2/scale))
            for t in tracks:
                col = _id_colour(t.track_id)
                x1, y1, x2, y2 = t.box
                cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
                cv2.putText(frame, f"ID {t.track_id}  {t.conf:.0%}",
                            (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX,
                            0.55, col, 2)
        else:
            tracks = tracker.update(frame)

        if tracks:
            log.person_detected(count=len(tracks))

        # ── 3. Face recognition (GPU, throttled) ─────────────
        if tracks and now - last_recog_time >= RECOGNITION_EVERY_S:
            faces, frame = recognizer.identify(frame)
            for f in faces:
                name = f["name"]
                if now - _seen_names.get(name, 0) > 10:
                    if name == "Unknown": log.unknown_face()
                    else:                 log.face_recognized(name)
                    _seen_names[name] = now
            last_recog_time = now

        # ── 4. LLM narration (async thread) ──────────────────
        force = _force_narrate.is_set()
        if force: _force_narrate.clear()
        if tracks: narrator.maybe_narrate(frame, tracks, force=force)

        # ── 5. Summary trigger ────────────────────────────────
        if _gen_summary.is_set():
            _gen_summary.clear()
            threading.Thread(target=narrator.generate_summary, daemon=True).start()

        # ── 6. HUD overlay ────────────────────────────────────
        elapsed = now - t_start
        fps     = frame_count / elapsed if elapsed > 0 else 0
        cv2.putText(frame,
                    f"Fazlerasheed Vision | {fps:.1f} fps | {DEVICE.upper()}",
                    (10, frame.shape[0] - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 180, 180), 1)

        # ── 7. Update display ─────────────────────────────────
        img_widget.value  = frame_to_jpeg_bytes(frame)
        status_label.value = (f"Status: Running | Frame #{frame_count} | "
                               f"Tracked: {len(tracks)}")
        fps_label.value    = f"FPS: {fps:.1f}"

except KeyboardInterrupt:
    pass

finally:
    webcam_stop()
    log.close()
    print(f"\n[Main] Stopped after {frame_count} frames.")
    # Final summary
    summary = log.today_summary()
    if summary:
        print("\n=== Today's Event Summary ===")
        for k, v in summary.items():
            print(f"  {k:30s} : {v}")


## 10 · Event Summary
Run this cell any time — even during a running session.

In [ ]:
# ── View event summary (run any time) ─────────────────────
from logger import EventLogger as _EL
_log = _EL(db_path=DB_PATH)

print("=== Today's Event Summary ===")
s = _log.today_summary()
if s:
    for k, v in sorted(s.items()):
        print(f"  {k:30s} : {v}")
else:
    print("  No events today yet.")

print("\n=== Last 10 Events ===")
for row in _log.recent_events(10):
    pid   = f" ({row['person_id']})" if row["person_id"] else ""
    extra = f"  — {row['extra'][:70]}" if row["extra"] else ""
    print(f"  {row['timestamp']}  {row['event_type']}{pid}{extra}")

alerts = _log.today_high_priority()
if alerts:
    print(f"\n=== ⚠  High-Priority Alerts ({len(alerts)}) ===")
    for a in alerts:
        print(f"  {a['timestamp']}  {a['detail']}")

_log.close()
